In [536]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score



# LOADING DATASET

In [534]:
train = pd.read_csv("train.csv")
stores = pd.read_csv("stores.csv")
features = pd.read_csv("features.csv")

In [4]:
train.head(2)

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True


In [5]:
stores.head(2)

,Store,Type,Size
0,1,A,151315
1,2,A,202307


In [6]:
features.head(2)

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True


# MERGING THE DATASET

In [8]:
df = pd.merge(train, features, on=["Store", "Date"], how="left")


In [9]:
df.head(2)

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True


In [10]:
df.shape

(421570, 15)

In [11]:
df = pd.merge(df, stores, on="Store", how="left")


In [12]:
df.head(2)

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True,A,151315


In [13]:
df.shape

(421570, 17)

In [14]:
df.isnull().sum()

Store                0
Dept                 0
Date                 0
Weekly_Sales         0
IsHoliday_x          0
Temperature          0
Fuel_Price           0
MarkDown1       270889
MarkDown2       310322
MarkDown3       284479
MarkDown4       286603
MarkDown5       270138
CPI                  0
Unemployment         0
IsHoliday_y          0
Type                 0
Size                 0
dtype: int64

In [15]:
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["Day"] = df["Date"].dt.day

# CONVERT BOOLEAN HOLIDAY TO INTEGER

In [17]:
df["IsHoliday_y"] = df["IsHoliday_y"].astype(int)


# SORT DATE BY DATE

In [19]:
df=df.sort_values("Date")

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 421570 entries, 0 to 421569
Data columns (total 21 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         421570 non-null  int64         
 1   Dept          421570 non-null  int64         
 2   Date          421570 non-null  datetime64[ns]
 3   Weekly_Sales  421570 non-null  float64       
 4   IsHoliday_x   421570 non-null  bool          
 5   Temperature   421570 non-null  float64       
 6   Fuel_Price    421570 non-null  float64       
 7   MarkDown1     150681 non-null  float64       
 8   MarkDown2     111248 non-null  float64       
 9   MarkDown3     137091 non-null  float64       
 10  MarkDown4     134967 non-null  float64       
 11  MarkDown5     151432 non-null  float64       
 12  CPI           421570 non-null  float64       
 13  Unemployment  421570 non-null  float64       
 14  IsHoliday_y   421570 non-null  int32         
 15  Type          421570 n

In [21]:
df.head(2)

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,MarkDown5,CPI,Unemployment,IsHoliday_y,Type,Size,Year,Month,Week,Day
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,...,NaN,211.096358,8.106,0,A,151315,2010,2,5,5
277665,29,5,2010-02-05,15552.08,False,24.36,2.788,NaN,NaN,NaN,...,NaN,131.527903,10.064,0,B,93638,2010,2,5,5


In [22]:
df=df.drop(columns=['IsHoliday_x'])

In [23]:
df=df.rename(columns={'IsHoliday_y':'IsHoliday'})

# HANDLING MISSING VALUES

In [25]:
markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
df[markdown_cols] = df[markdown_cols].fillna(0)

In [26]:
print(df[markdown_cols].isnull().sum())

MarkDown1    0
MarkDown2    0
MarkDown3    0
MarkDown4    0
MarkDown5    0
dtype: int64


# Checking Dupicates

In [28]:
duplicates = df[df.duplicated(keep=False)]
print(duplicates)


Empty DataFrame
Columns: [Store, Dept, Date, Weekly_Sales, Temperature, Fuel_Price, MarkDown1, MarkDown2, MarkDown3, MarkDown4, MarkDown5, CPI, Unemployment, IsHoliday, Type, Size, Year, Month, Week, Day]
Index: []


In [29]:
df.head()

,Store,Dept,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday,Type,Size,Year,Month,Week,Day
0,1,1,2010-02-05,24924.50,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,0,A,151315,2010,2,5,5
277665,29,5,2010-02-05,15552.08,24.36,2.788,0.0,0.0,0.0,0.0,0.0,131.527903,10.064,0,B,93638,2010,2,5,5
277808,29,6,2010-02-05,3200.22,24.36,2.788,0.0,0.0,0.0,0.0,0.0,131.527903,10.064,0,B,93638,2010,2,5,5
277951,29,7,2010-02-05,10820.05,24.36,2.788,0.0,0.0,0.0,0.0,0.0,131.527903,10.064,0,B,93638,2010,2,5,5
278094,29,8,2010-02-05,20055.64,24.36,2.788,0.0,0.0,0.0,0.0,0.0,131.527903,10.064,0,B,93638,2010,2,5,5


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 421570 entries, 0 to 421569
Data columns (total 20 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         421570 non-null  int64         
 1   Dept          421570 non-null  int64         
 2   Date          421570 non-null  datetime64[ns]
 3   Weekly_Sales  421570 non-null  float64       
 4   Temperature   421570 non-null  float64       
 5   Fuel_Price    421570 non-null  float64       
 6   MarkDown1     421570 non-null  float64       
 7   MarkDown2     421570 non-null  float64       
 8   MarkDown3     421570 non-null  float64       
 9   MarkDown4     421570 non-null  float64       
 10  MarkDown5     421570 non-null  float64       
 11  CPI           421570 non-null  float64       
 12  Unemployment  421570 non-null  float64       
 13  IsHoliday     421570 non-null  int32         
 14  Type          421570 non-null  object        
 15  Size          421570 n

## Aggregation of weekly_sales on store

In [454]:
df = df.groupby(['Store', 'Date'], as_index=False).agg({
    'Weekly_Sales': 'sum',
    'IsHoliday': 'first',
    'Temperature': 'mean',
    'Fuel_Price': 'sum',
    'MarkDown1': 'mean',
    'MarkDown2': 'mean',
    'MarkDown3': 'mean',
    'MarkDown4': 'mean',
    'MarkDown5': 'mean',
    'CPI': 'mean',
    'Unemployment': 'mean',
    'Type': 'first',
    'Size': 'first',
    'Year': 'first',
    'Month': 'first',
    'Week': 'first',
    'Day': 'first'
})

In [456]:
df.tail(10)

,Store,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,Year,Month,Week,Day
6425,45,2012-08-24,718232.26,0,72.62,3.834,531725.40,3911.46,1474.00,369710.69,153561.99,191.344887,8.684,B,118221,2012,8,34,24
6426,45,2012-08-31,734297.87,0,75.09,3.867,1607608.40,408.00,6319.24,475205.08,271464.84,191.461281,8.684,B,118221,2012,8,35,31
6427,45,2012-09-07,766512.66,1,75.70,3.911,760687.05,883.20,3631.47,127979.13,141843.30,191.577676,8.684,B,118221,2012,9,36,7
6428,45,2012-09-14,702238.27,0,67.87,3.948,787148.55,0.00,296.70,236098.68,363555.48,191.699850,8.684,B,118221,2012,9,37,14
6429,45,2012-09-21,723086.20,0,65.32,4.038,566297.40,6182.76,4237.08,159217.46,580916.80,191.856704,8.684,B,118221,2012,9,38,21
6430,45,2012-09-28,713173.95,0,64.88,3.997,300736.26,1362.24,99.00,105666.66,217024.50,192.013558,8.684,B,118221,2012,9,39,28
6431,45,2012-10-05,733455.07,0,64.89,3.985,343178.32,0.00,1279.76,153233.24,159120.68,192.170412,8.667,B,118221,2012,10,40,5
6432,45,2012-10-12,734464.36,0,54.47,4.000,129114.48,0.00,520.74,39555.12,263375.64,192.327265,8.667,B,118221,2012,10,41,12
6433,45,2012-10-19,718125.53,0,56.47,3.969,132265.32,0.00,209.88,28890.18,101474.34,192.330854,8.667,B,118221,2012,10,42,19
6434,45,2012-10-26,760281.43,0,58.85,3.882,269266.97,3891.36,6700.00,14199.98,57508.11,192.308899,8.667,B,118221,2012,10,43,26


In [458]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         6435 non-null   int64         
 1   Date          6435 non-null   datetime64[ns]
 2   Weekly_Sales  6435 non-null   float64       
 3   IsHoliday     6435 non-null   int32         
 4   Temperature   6435 non-null   float64       
 5   Fuel_Price    6435 non-null   float64       
 6   MarkDown1     6435 non-null   float64       
 7   MarkDown2     6435 non-null   float64       
 8   MarkDown3     6435 non-null   float64       
 9   MarkDown4     6435 non-null   float64       
 10  MarkDown5     6435 non-null   float64       
 11  CPI           6435 non-null   float64       
 12  Unemployment  6435 non-null   float64       
 13  Type          6435 non-null   object        
 14  Size          6435 non-null   int64         
 15  Year          6435 non-null   int32   

In [462]:
from sklearn.model_selection import train_test_split
features = ['Store' ,'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'IsHoliday','MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5','Size']  # choose based on EDA
X = df[features]
y = df['Weekly_Sales'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



## Feature Scaling

In [525]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train)
test_scaled=scaler.transform(X_test)

In [529]:
train_scaled

array([[-1.16466588,  0.8057783 , -1.71088305, ..., -0.27708288,
        -0.37246616,  0.39298496],
       [-1.24165623, -2.15108453,  0.01856676, ..., -0.27708288,
        -0.37246616, -0.94354094],
       [ 1.45300579,  0.04355515, -0.51121407, ..., -0.27708288,
        -0.37246616, -1.43501303],
       ...,
       [ 1.06805407,  1.42237786,  0.70158997, ..., -0.27708288,
        -0.37246616, -1.43152775],
       [ 1.14504441, -0.85508863,  0.14116067, ..., -0.27167184,
        -0.23351649, -1.43501303],
       [-1.24165623, -1.79487513, -1.78093671, ..., -0.27708288,
        -0.37246616, -0.94354094]])

In [469]:
x_train=pd.DataFrame(train_scaled, columns=X_train.columns)
x_test=pd.DataFrame(test_scaled, columns=X_test.columns)

In [471]:
x_train

,Store,Temperature,Fuel_Price,CPI,Unemployment,IsHoliday,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,Size
0,-1.164666,0.805778,-1.710883,1.103804,-0.897874,-0.268427,-0.424271,-0.163740,-0.07971,-0.277083,-0.372466,0.392985
1,-1.241656,-2.151085,0.018567,0.522548,0.438475,-0.268427,-0.424271,-0.163740,-0.07971,-0.277083,-0.372466,-0.943541
2,1.453006,0.043555,-0.511214,-1.146134,0.537246,-0.268427,-0.424271,-0.163740,-0.07971,-0.277083,-0.372466,-1.435013
3,-0.625733,0.473930,1.187587,-0.899892,-0.101831,-0.268427,-0.424271,-0.163740,-0.07971,-0.277083,-0.372466,-0.103525
4,1.453006,1.459731,0.830751,-1.077504,0.138957,-0.268427,-0.424271,-0.163740,-0.07971,-0.277083,-0.372466,-1.435013
...,...,...,...,...,...,...,...,...,...,...,...,...
5143,0.298151,-1.143088,0.123647,-0.873464,-0.090619,-0.268427,-0.424271,-0.163740,-0.07971,-0.277083,-0.372466,1.170931
5144,1.068054,-0.324564,-1.435047,0.989125,0.255881,-0.268427,-0.424271,-0.163740,-0.07971,-0.277083,-0.372466,-1.431528
5145,1.068054,1.422378,0.701590,1.088999,0.096246,-0.268427,-0.424271,-0.163740,-0.07971,-0.277083,-0.372466,-1.431528
5146,1.145044,-0.855089,0.141161,-1.056351,2.612512,3.725401,-0.376912,0.117311,-0.07971,-0.271672,-0.233516,-1.435013


In [473]:
x_test

,Store,Temperature,Fuel_Price,CPI,Unemployment,IsHoliday,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,Size
0,-0.394762,-0.976893,-1.222697,-1.012720,0.643492,-0.268427,-0.424271,-0.163740,-0.079710,-0.277083,-0.372466,-0.152382
1,0.067180,0.457148,1.314560,-0.924048,0.114932,-0.268427,-0.424271,-0.163740,-0.079710,-0.277083,-0.372466,1.165149
2,-1.626608,0.294201,0.018567,1.166123,-0.296704,-0.268427,-0.424271,-0.163740,-0.079710,-0.277083,-0.372466,1.141195
3,0.221160,-1.101945,0.452024,-0.893570,-0.212882,-0.268427,0.810373,1.125505,-0.028067,0.158595,0.182767,0.352350
4,0.991064,-0.047392,-1.450371,0.979040,0.297525,-0.268427,-0.424271,-0.163740,-0.079710,-0.277083,-0.372466,-1.431528
...,...,...,...,...,...,...,...,...,...,...,...,...
1282,-0.317772,0.745147,-0.922779,-0.991574,0.054601,-0.268427,-0.424271,-0.163740,-0.079710,-0.277083,-0.372466,1.165149
1283,1.683977,0.478261,-1.016914,0.267528,0.481721,-0.268427,-0.424271,-0.163740,-0.079710,-0.277083,-0.372466,-0.190910
1284,1.683977,-1.793251,-1.268669,0.263065,0.531373,-0.268427,-0.424271,-0.163740,-0.079710,-0.277083,-0.372466,-0.190910
1285,-1.626608,0.153450,-1.410966,0.991765,0.108525,-0.268427,-0.424271,-0.163740,-0.079710,-0.277083,-0.372466,1.141195


## Linear Regression

In [476]:
model = LinearRegression()
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
print("R-squared:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

coefficients = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
print(coefficients)

R-squared: 0.6979258816177538
RMSE: 311952.8106854486
                Coefficient
Store         -88309.806725
Temperature    26443.390410
Fuel_Price    -18739.814729
CPI           -69081.766729
Unemployment  -22412.417941
IsHoliday      11592.836354
MarkDown1       2243.819314
MarkDown2       1948.688062
MarkDown3      46752.389661
MarkDown4       5215.947800
MarkDown5      25644.818722
Size          427161.784512


In [478]:
from sklearn.ensemble import RandomForestRegressor

In [480]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)



RandomForestRegressor(random_state=42)

In [173]:
y_pred = model.predict(x_test)

In [174]:
print("R-squared_RFR:", r2_score(y_test, y_pred))
print("RMSE_RFR:", np.sqrt(mean_squared_error(y_test, y_pred)))


R-squared_RFR: 0.9432038025400018
RMSE_RFR: 135266.97403866643


In [178]:
# Feature importance
importances = pd.DataFrame(model.feature_importances_, index=X.columns, columns=["Importance"])
print(importances.sort_values("Importance", ascending=False))

              Importance
Size            0.701844
Store           0.145809
CPI             0.072476
MarkDown3       0.020589
Unemployment    0.020001
Fuel_Price      0.014907
Temperature     0.014794
MarkDown4       0.002424
MarkDown1       0.002210
MarkDown5       0.002077
IsHoliday       0.001711
MarkDown2       0.001159


In [44]:
from sklearn.ensemble import HistGradientBoostingRegressor
model = HistGradientBoostingRegressor( max_iter=200,
    learning_rate=0.05,
    max_depth=8,
    random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [45]:
print("R-squared_HGBR:", r2_score(y_test, y_pred))
print("RMSE_HGBR:", np.sqrt(mean_squared_error(y_test, y_pred))) 

R-squared_HGBR: 0.9533744105633755
RMSE_HGBR: 122558.74699811089


In [46]:
coefficients = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
print(coefficients)

AttributeError: 'HistGradientBoostingRegressor' object has no attribute 'coef_'

## Decision Tree Regressor

In [490]:
from sklearn.tree import DecisionTreeRegressor

In [492]:
model = DecisionTreeRegressor(splitter='best',max_depth=None,min_samples_split=5)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)


In [494]:
print("R-squared:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R-squared: 0.9079021414399456
RMSE: 172249.0799049837


In [496]:
importances = pd.DataFrame(model.feature_importances_, index=X.columns, columns=["Importance"])
print(importances.sort_values("Importance", ascending=False))

              Importance
Size            0.698716
Store           0.148373
CPI             0.084641
MarkDown3       0.019212
Unemployment    0.016492
Fuel_Price      0.014430
Temperature     0.010622
MarkDown1       0.002235
MarkDown4       0.002189
MarkDown5       0.001613
IsHoliday       0.000893
MarkDown2       0.000584


## RandomForestRegressor

In [499]:
from sklearn.ensemble import RandomForestRegressor

In [501]:
model = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42)
model.fit(x_train, y_train)

RandomForestRegressor(random_state=42)

In [502]:
y_pred = model.predict(x_test)

In [503]:
print("R-squared_RFR:", r2_score(y_test, y_pred))
print("RMSE_RFR:", np.sqrt(mean_squared_error(y_test, y_pred)))


R-squared_RFR: 0.9432038025400018
RMSE_RFR: 135266.97403866643


In [507]:
importances = pd.DataFrame(model.feature_importances_, index=X.columns, columns=["Importance"])
print(importances.sort_values("Importance", ascending=False))

              Importance
Size            0.701844
Store           0.145809
CPI             0.072476
MarkDown3       0.020589
Unemployment    0.020001
Fuel_Price      0.014907
Temperature     0.014794
MarkDown4       0.002424
MarkDown1       0.002210
MarkDown5       0.002077
IsHoliday       0.001711
MarkDown2       0.001159


##  Bayesian Ridge

In [516]:
from sklearn.linear_model import BayesianRidge

In [518]:
model = BayesianRidge()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

In [520]:
print("R-squared:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


R-squared: 0.6979161931736009
RMSE: 311957.8132874049
